<a href="https://colab.research.google.com/github/ishjeetsingh/india-solar-soiling/blob/main/02_site_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests, pandas as pd, time

sites = pd.read_csv("https://raw.githubusercontent.com/ishjeetsingh/india-solar-soiling/main/data/soiling_sites.csv")
sites = sites.dropna(subset=["lat", "lon"])   # site 7 has no coordinates yet

def get_power(lat, lon, start="20240101", end="20241231"):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {"parameters": "PRECTOTCORR,ALLSKY_SFC_SW_DWN,AOD_55",
              "community": "RE", "latitude": lat, "longitude": lon,
              "start": start, "end": end, "format": "JSON"}
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["properties"]["parameter"])
    df.index = pd.to_datetime(df.index)
    df = df.replace(-999, pd.NA)
    return df.rename(columns={"PRECTOTCORR": "rain_mm",
                              "ALLSKY_SFC_SW_DWN": "irradiance_kwh_m2",
                              "AOD_55": "aod_merra2"})

frames = []
for _, s in sites.iterrows():
    df = get_power(s.lat, s.lon)
    df["site"] = s.site
    frames.append(df)
    time.sleep(1)   # be polite to NASA's server

daily = pd.concat(frames).rename_axis("date").reset_index()
daily.to_csv("site_daily_2024.csv", index=False)
print(daily.groupby("site")[["rain_mm", "irradiance_kwh_m2", "aod_merra2"]].mean().round(2))


                      rain_mm  irradiance_kwh_m2  aod_merra2
site                                                        
Chennai                  4.44               5.00        0.51
Delhi                    2.26               4.59        0.76
Gandhinagar              2.73               5.15        0.57
Gandhinagar seasonal     2.73               5.15        0.57
Lucknow                  3.18               4.50        0.91
Mumbai IIT Bombay        8.56               5.02        0.64
Pune                     7.14               4.86        0.68


In [2]:
import numpy as np
from scipy import stats

daily = pd.read_csv("site_daily_2024.csv", parse_dates=["date"])
dry = daily[daily.rain_mm < 1]                               # dry days only
aod = dry.groupby("site").aod_merra2.mean().rename("aod_dry")
df = sites.set_index("site").join(aod).dropna(subset=["aod_dry", "rate_pct_per_day"])

x, y = df.aod_dry.values, df.rate_pct_per_day.values
k = (x * y).sum() / (x * x).sum()                            # best k for: rate = k × AOD
df["pred"] = k * x

r, p = stats.pearsonr(x, y)
rmse = np.sqrt(((df.pred - y) ** 2).mean())
rmse_base = np.sqrt(((y - y.mean()) ** 2).mean())            # "just guess the average" baseline
loo = [(np.delete(x, i) * np.delete(y, i)).sum() / (np.delete(x, i) ** 2).sum() * x[i] for i in range(len(x))]
rmse_loo = np.sqrt(((np.array(loo) - y) ** 2).mean())

print(df[["aod_dry", "rate_pct_per_day", "pred"]].round(3))
print(f"k = {k:.3f} | r = {r:.2f} (p = {p:.2f}) | RMSE = {rmse:.3f} | LOO RMSE = {rmse_loo:.3f} | baseline RMSE = {rmse_base:.3f}")

                      aod_dry  rate_pct_per_day   pred
site                                                  
Gandhinagar             0.462             0.370  0.304
Gandhinagar seasonal    0.462             0.450  0.304
Delhi                   0.654             0.450  0.430
Chennai                 0.480             0.255  0.316
Mumbai IIT Bombay       0.483             0.400  0.318
Lucknow                 0.820             0.390  0.540
Pune                    0.511             0.340  0.336
k = 0.658 | r = 0.25 (p = 0.59) | RMSE = 0.092 | LOO RMSE = 0.114 | baseline RMSE = 0.063


In [3]:
!pip -q install earthaccess
import earthaccess, xarray as xr
earthaccess.login(strategy="interactive")
res = earthaccess.search_data(short_name="M2TMNXADG", temporal=("2024-01-01", "2024-12-31"))
print(len(res), "files found")
files = earthaccess.download(res, "merra2")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.9.0 which is incompatible.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.
Enter your Earthdata Login username: ishjeetsingh
Enter your Earthdata password: ··········
12 files found


QUEUEING TASKS | :   0%|          | 0/12 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/12 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/12 [00:00<?, ?it/s]

In [6]:
import glob
parts = []
for f in sorted(glob.glob("merra2/*.nc4")):
    d = xr.open_dataset(f)
    dv = [v for v in d.data_vars if v.startswith(("DUDP", "DUSD"))]
    parts.append(d[dv].sel(lat=slice(5, 38), lon=slice(66, 99)).load())
    print(f[-14:], "ok")
ds = xr.concat(parts, dim="time")
dvars = list(ds.data_vars)
dep = sum(ds[v] for v in dvars) * 86400 * 1e6

rows = []
for _, s in sites.iterrows():
    ts = dep.sel(lat=s.lat, lon=s.lon, method="nearest").to_series()
    ts.index = ts.index.to_period("M")
    rows.append(ts.rename(s.site))
dep_m = pd.concat(rows, axis=1)

rain_m = daily.set_index("date").groupby("site").rain_mm.resample("M").sum().unstack(0)
rain_m.index = rain_m.index.to_period("M")
dust_dry = dep_m.where(rain_m < 30).mean()                            # dry months only (<30 mm)

def evaluate(proxy, name):
    d = sites.set_index("site").join(proxy.rename("x")).dropna(subset=["x", "rate_pct_per_day"])
    x, y = d.x.values, d.rate_pct_per_day.values
    r, p = stats.pearsonr(x, y)
    loo = [(np.delete(x, i) * np.delete(y, i)).sum() / (np.delete(x, i) ** 2).sum() * x[i] for i in range(len(x))]
    rmse_loo = np.sqrt(((np.array(loo) - y) ** 2).mean())
    print(f"{name}: r = {r:.2f} (p = {p:.2f}) | LOO RMSE = {rmse_loo:.3f} | baseline = {np.sqrt(((y - y.mean()) ** 2).mean()):.3f}")

print(dvars)
print(dust_dry.round(2))
evaluate(aod, "MERRA-2 AOD")
evaluate(dust_dry, "MERRA-2 dust deposition")

_Nx.202401.nc4 ok
_Nx.202402.nc4 ok
_Nx.202403.nc4 ok
_Nx.202404.nc4 ok
_Nx.202405.nc4 ok
_Nx.202406.nc4 ok
_Nx.202407.nc4 ok
_Nx.202408.nc4 ok
_Nx.202409.nc4 ok
_Nx.202410.nc4 ok
_Nx.202411.nc4 ok
_Nx.202412.nc4 ok
['DUDP001', 'DUDP002', 'DUDP003', 'DUDP004', 'DUDP005', 'DUSD001', 'DUSD002', 'DUSD003', 'DUSD004', 'DUSD005']
Gandhinagar             17.76
Gandhinagar seasonal    17.76
Delhi                   14.56
Chennai                  2.45
Mumbai IIT Bombay        5.62
Lucknow                 10.70
Pune                     4.21
dtype: float32
MERRA-2 AOD: r = 0.25 (p = 0.59) | LOO RMSE = 0.114 | baseline = 0.063
MERRA-2 dust deposition: r = 0.72 (p = 0.07) | LOO RMSE = 0.172 | baseline = 0.063


/tmp/ipykernel_11501/2819380755.py:19: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rain_m = daily.set_index("date").groupby("site").rain_mm.resample("M").sum().unstack(0)


In [7]:
def evaluate2(proxy, name):
    d = sites.set_index("site").join(proxy.rename("x")).dropna(subset=["x", "rate_pct_per_day"])
    x, y = d.x.values, d.rate_pct_per_day.values
    b, a = np.polyfit(x, y, 1)                      # straight line with an intercept
    loo = []
    for i in range(len(x)):
        bi, ai = np.polyfit(np.delete(x, i), np.delete(y, i), 1)
        loo.append(ai + bi * x[i])
    rmse_loo = np.sqrt(((np.array(loo) - y) ** 2).mean())
    base_loo = np.sqrt(np.mean([(y[i] - np.delete(y, i).mean()) ** 2 for i in range(len(y))]))
    print(f"{name}: rate = {a:.3f} + {b:.4f}·x | LOO RMSE = {rmse_loo:.3f} | LOO baseline = {base_loo:.3f}")

evaluate2(aod, "MERRA-2 AOD")
evaluate2(dust_dry, "MERRA-2 dust deposition")


MERRA-2 AOD: rate = 0.311 + 0.1235·x | LOO RMSE = 0.086 | LOO baseline = 0.073
MERRA-2 dust deposition: rate = 0.301 + 0.0075·x | LOO RMSE = 0.066 | LOO baseline = 0.073
